In [1]:
import pybroker
from pybroker.ext.data import AKShare
from pybroker import ExecContext, StrategyConfig, Strategy
from pybroker.data import DataSource
import matplotlib.pyplot as plt
from datetime import datetime
import riskfolio as rp
import akshare as ak
import pandas as pd
import numpy as np
import sqlite3
import datetime

import talib
from pybroker.vect import cross

# 导入Numba库
from numba import njit

#正常显示画图时出现的中文和负号
from pylab import mpl

mpl.rcParams['font.sans-serif']=['SimHei']
mpl.rcParams['axes.unicode_minus']=False

akshare = AKShare()

pybroker.enable_data_source_cache('akshare')

In [2]:
# 读取本地CSV数据
df = pd.read_csv(r"C:\Users\Administrator\OneDrive\同步\iFinD data\future_data_merged_all.csv")

# 转换时间列格式
df['time'] = pd.to_datetime(df['time'])

df.columns

Index(['time', 'thscode', 'open', 'close', 'high', 'low'], dtype='object')

In [3]:
# 检查列名并转换为pybroker所需格式
if 'thscode' in df.columns:
    df = df.rename(columns={'thscode': 'symbol'})
else:
    raise ValueError("数据中不存在 'thscode' 列，请检查数据格式")

# 添加volume列（如果不存在），全部设置为1000000
if 'volume' not in df.columns:
    df['volume'] = 1000000

# 确保包含pybroker必需的列
required_columns = {'symbol', 'time', 'open', 'high', 'low', 'close', 'volume'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"数据缺少以下必需列: {', '.join(missing_columns)}")

df

,time,symbol,open,close,high,low,volume
0,2022-01-04 09:05:00,AGZL.SHF,4836.0,4838.0,4861.0,4830.0,1000000
1,2022-01-04 09:10:00,AGZL.SHF,4838.0,4840.0,4840.0,4831.0,1000000
2,2022-01-04 09:15:00,AGZL.SHF,4841.0,4822.0,4845.0,4818.0,1000000
3,2022-01-04 09:20:00,AGZL.SHF,4822.0,4819.0,4822.0,4814.0,1000000
4,2022-01-04 09:25:00,AGZL.SHF,4818.0,4814.0,4818.0,4806.0,1000000
...,...,...,...,...,...,...,...
2355044,2025-05-30 14:40:00,ZNZL.SHF,22255.0,22245.0,22260.0,22235.0,1000000
2355045,2025-05-30 14:45:00,ZNZL.SHF,22245.0,22240.0,22250.0,22235.0,1000000
2355046,2025-05-30 14:50:00,ZNZL.SHF,22240.0,22195.0,22245.0,22190.0,1000000
2355047,2025-05-30 14:55:00,ZNZL.SHF,22195.0,22235.0,22245.0,22195.0,1000000


In [5]:
# 检查数据是否按时间升序排列
if not df['time'].is_monotonic_increasing:
    print("警告：数据未按时间升序排列，需要排序！")
    df = df.sort_values('time')  # 重新按时间排序

In [6]:

def compute_bb(df):
    # 检查并转换必要的列
    required_columns = ['close', 'open', 'high', 'low']

    # 确保所有必需的列都存在
    for col in required_columns:
        if col not in group_df.columns:
            raise ValueError(f"Missing required column: {col}")

    # 转换数据类型
    for col in required_columns:
        group_df[col] = group_df[col].astype(float)

    # 计算MACD指标
    macd, macdsignal, macdhist = talib.MACD(group_df['close'], fastperiod=12, slowperiod=26, signalperiod=9)

    # 计算布林带指标
    upper, middle, lower = talib.BBANDS(group_df['close'], timeperiod=20)

    # 计算KD指标
    k, d = talib.STOCH(group_df['high'], group_df['low'], group_df['close'],
                        fastk_period=14, slowk_period=3, slowd_period=3)

    # 创建包含所有指标的DataFrame
    indicators_df = pd.DataFrame({
        'macd': macd,
        'macdsignal': macdsignal,
        'macdhist': macdhist,
        'bb_high': upper,
        'bb_mid': middle,
        'bb_low': lower,
        'kd_k': k,
        'kd_d': d
    })

    return indicators_df

# 修正groupby操作 - 直接对整个DataFrame分组并应用函数
df_computed = df_sorted.groupby('thscode', group_keys=False).apply(compute_bb)

# 将计算结果合并回原DataFrame
df_sorted = pd.concat([df_sorted.reset_index(drop=True), df_computed.reset_index(drop=True)], axis=1)

# 重命名列以符合pybroker要求
df_sorted = df_sorted.rename(columns={'time': 'date', 'thscode': 'symbol'})

NameError: name 'df_sorted' is not defined

In [18]:
df_sorted

,date,symbol,open,close,high,low,macd,macdsignal,macdhist,bb_high,bb_mid,bb_low,kd_k,kd_d
0,2022-01-04 09:05:00,AGZL.SHF,4836.0,4838.0,4861.0,4830.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-01-04 09:10:00,AGZL.SHF,4838.0,4840.0,4840.0,4831.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-01-04 09:15:00,AGZL.SHF,4841.0,4822.0,4845.0,4818.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-04 09:20:00,AGZL.SHF,4822.0,4819.0,4822.0,4814.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-01-04 09:25:00,AGZL.SHF,4818.0,4814.0,4818.0,4806.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2355044,2025-05-30 14:40:00,ZNZL.SHF,22255.0,22245.0,22260.0,22235.0,-6.641569,-11.538710,4.897140,22287.015551,22232.75,22178.484449,65.952381,49.523810
2355045,2025-05-30 14:45:00,ZNZL.SHF,22245.0,22240.0,22250.0,22235.0,-5.131194,-10.257206,5.126013,22283.854560,22231.50,22179.145440,66.917293,61.181426
2355046,2025-05-30 14:50:00,ZNZL.SHF,22240.0,22195.0,22245.0,22190.0,-7.479119,-9.701589,2.222470,22279.274266,22227.75,22176.225734,43.372320,58.747331
2355047,2025-05-30 14:55:00,ZNZL.SHF,22195.0,22235.0,22245.0,22195.0,-6.042547,-8.969781,2.927234,22268.127717,22225.00,22181.872283,37.816764,49.368792


In [47]:
# 定义交易策略
def kd_strategy(ctx):
    # 获取 KD 指标
    kd_k = ctx._col_scope.fetch(ctx.symbol, 'kd_k', ctx._sym_end_index[ctx.symbol])
    kd_d = ctx._col_scope.fetch(ctx.symbol, 'kd_d', ctx._sym_end_index[ctx.symbol])
    symbol = ctx.symbol
    # 确保 ctx 对象有 portfolio 属性
    position = ctx.portfolio.position(symbol) if hasattr(ctx, 'portfolio') else None

    if position is not None:
        # 开仓条件
        if kd_k < 20 and kd_d < 20 and kd_k > kd_d and position.shares == 0:
            # 计算最大仓位
            eligible_symbols = [sym for sym in ctx._scope.get_all_symbols() if 
                                ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) < 20 and 
                                ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) < 20 and 
                                ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) > ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) and 
                                ctx.portfolio.position(sym).shares == 0]
            max_position_per_symbol = len(eligible_symbols) / 0.8
            # 开仓
            ctx.buy(max_position_per_symbol)
        elif kd_k > 80 and kd_d > 80 and kd_k < kd_d and position.shares == 0:
            # 计算最大仓位
            eligible_symbols = [sym for sym in ctx._scope.get_all_symbols() if 
                                ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) > 80 and 
                                ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) > 80 and 
                                ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) < ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) and 
                                ctx.portfolio.position(sym).shares == 0]
            max_position_per_symbol = len(eligible_symbols) / 0.8
            # 开仓
            ctx.sell(max_position_per_symbol)

        # 止损条件
        if position.shares > 0 and kd_k < 20 and kd_d < 20 and kd_k < kd_d:
            ctx.sell(position.shares)
        elif position.shares < 0 and kd_k > 80 and kd_d > 80 and kd_k > kd_d:
            ctx.buy(-position.shares)

        # 止盈条件
        if position.shares > 0 and kd_k > 80 and kd_d > 80:
            ctx.sell(position.shares)
        elif position.shares < 0 and kd_k > 20 and kd_d > 20:
            ctx.buy(-position.shares)

# 配置策略
config = StrategyConfig(initial_cash=10000000)
strategy = Strategy(df_sorted, start_date=df_sorted['date'].min(), end_date=df_sorted['date'].max(), config=config)

# 添加执行函数和交易品种
strategy.add_execution(kd_strategy, symbols=df_sorted['symbol'].unique().tolist())

# 运行策略
result = strategy.backtest()

Backtesting: 2022-01-04 09:05:00 to 2025-05-30 15:00:00

Test split: 2022-01-04 09:05:00 to 2025-05-30 15:00:00


  0% (0 of 89707) |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
  0% (1 of 89707) |                      | Elapsed Time: 0:00:00 ETA:   3:58:52
  0% (761 of 89707) |                    | Elapsed Time: 0:00:00 ETA:   0:00:30
  1% (1591 of 89707) |                   | Elapsed Time: 0:00:00 ETA:   0:00:19
  2% (2281 of 89707) |                   | Elapsed Time: 0:00:00 ETA:   0:00:16
  3% (3141 of 89707) |                   | Elapsed Time: 0:00:00 ETA:   0:00:16
  4% (4031 of 89707) |                   | Elapsed Time: 0:00:00 ETA:   0:00:15
  5% (4551 of 89707) |                   | Elapsed Time: 0:00:00 ETA:   0:00:14
  6% (5401 of 89707) |#                  | Elapsed Time: 0:00:00 ETA:   0:00:13
  7% (6301 of 89707) |#                  | Elapsed Time: 0:00:00 ETA:   0:00:12
  7% (6821 of 89707) |#                  | Elapsed Time: 0:00:01 ETA:   0:00:12
  8% (7571 of 89707) |#                  | Elapsed Time: 0:00:01 ETA:   0:00:12
  9% (8401 of 89707) |#                 


Finished backtest: 0:00:13


In [48]:
result.orders

,type,symbol,date,shares,limit_price,fill_price,fees
id,,,,,,,


In [45]:
result.metrics_df

,name,value
0,trade_count,0.0
1,initial_market_value,1000000.0
2,end_market_value,1000000.0
3,total_pnl,0.0
4,unrealized_pnl,0.0
5,total_return_pct,0.0
6,total_profit,0.0
7,total_loss,0.0
8,total_fees,0.0
9,max_drawdown,-0.0


In [ ]:
import pybroker
from pybroker import ExecContext, StrategyConfig, Strategy
import pandas as pd
import talib

# 读取数据
df = pd.read_csv(r"C:\Users\Administrator\OneDrive\同步\iFinD data\future_data_merged_all.csv")
df['time'] = pd.to_datetime(df['time'])
df_sorted = df.sort_values(by=['thscode', 'time'])

def compute_bb(group_df):
    # 检查并转换必要的列
    required_columns = ['close', 'open', 'high', 'low']

    # 确保所有必需的列都存在
    for col in required_columns:
        if col not in group_df.columns:
            raise ValueError(f"Missing required column: {col}")

    # 转换数据类型
    for col in required_columns:
        group_df[col] = group_df[col].astype(float)

    # 计算MACD指标
    macd, macdsignal, macdhist = talib.MACD(group_df['close'], fastperiod=12, slowperiod=26, signalperiod=9)

    # 计算布林带指标
    upper, middle, lower = talib.BBANDS(group_df['close'], timeperiod=20)

    # 计算KD指标
    k, d = talib.STOCH(group_df['high'], group_df['low'], group_df['close'],
                        fastk_period=14, slowk_period=3, slowd_period=3)

    # 创建包含所有指标的DataFrame
    indicators_df = pd.DataFrame({
        'macd': macd,
        'macdsignal': macdsignal,
        'macdhist': macdhist,
        'bb_high': upper,
        'bb_mid': middle,
        'bb_low': lower,
        'kd_k': k,
        'kd_d': d
    })

    return indicators_df

# 修正groupby操作 - 直接对整个DataFrame分组并应用函数
df_computed = df_sorted.groupby('thscode', group_keys=False).apply(compute_bb)

# 将计算结果合并回原DataFrame
df_sorted = pd.concat([df_sorted.reset_index(drop=True), df_computed.reset_index(drop=True)], axis=1)

# 重命名列以符合pybroker要求
df_sorted = df_sorted.rename(columns={'time': 'date', 'thscode': 'symbol'})


C:\Users\Administrator\AppData\Local\Temp\ipykernel_69764\902902078.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_computed = df_sorted.groupby('thscode', group_keys=False).apply(compute_bb)


In [35]:
# 定义交易策略
def kd_strategy(ctx: ExecContext):
    # 获取 KD 指标
    kd_k = ctx._col_scope.fetch(ctx.symbol, 'kd_k', ctx._sym_end_index[ctx.symbol])
    kd_d = ctx._col_scope.fetch(ctx.symbol, 'kd_d', ctx._sym_end_index[ctx.symbol])
    symbol = ctx.symbol
    position = ctx.portfolio.position(symbol)

    # 开仓条件
    if kd_k < 20 and kd_d < 20 and kd_k > kd_d and position.shares == 0:
        # 计算最大仓位
        eligible_symbols = [sym for sym in ctx._scope.get_all_symbols() if 
                            ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) < 20 and 
                            ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) < 20 and 
                            ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) > ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) and 
                            ctx.portfolio.position(sym).shares == 0]
        max_position_per_symbol = len(eligible_symbols) / 0.8
        # 开仓
        ctx.buy(max_position_per_symbol)
    elif kd_k > 80 and kd_d > 80 and kd_k < kd_d and position.shares == 0:
        # 计算最大仓位
        eligible_symbols = [sym for sym in ctx._scope.get_all_symbols() if 
                            ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) > 80 and 
                            ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) > 80 and 
                            ctx._col_scope.fetch(sym, 'kd_k', ctx._sym_end_index[sym]) < ctx._col_scope.fetch(sym, 'kd_d', ctx._sym_end_index[sym]) and 
                            ctx.portfolio.position(sym).shares == 0]
        max_position_per_symbol = len(eligible_symbols) / 0.8
        # 开仓
        ctx.sell(max_position_per_symbol)

    # 止损条件
    if position.shares > 0 and kd_k < 20 and kd_d < 20 and kd_k < kd_d:
        ctx.sell(position.shares)
    elif position.shares < 0 and kd_k > 80 and kd_d > 80 and kd_k > kd_d:
        ctx.buy(-position.shares)

    # 止盈条件
    if position.shares > 0 and kd_k > 80 and kd_d > 80:
        ctx.sell(position.shares)
    elif position.shares < 0 and kd_k > 20 and kd_d > 20:
        ctx.buy(-position.shares)

In [36]:
# 配置策略
config = StrategyConfig(initial_cash=1000000)
strategy = Strategy(df_sorted, start_date=df_sorted['date'].min(), end_date=df_sorted['date'].max(), config=config)

# 添加执行函数和交易品种
strategy.add_execution(kd_strategy, symbols=df_sorted['symbol'].unique().tolist())

# 运行策略
result = strategy.backtest()

# 打印回测结果
print(result.metrics())

Backtesting: 2022-01-04 09:05:00 to 2025-05-30 15:00:00

Test split: 2022-01-04 09:05:00 to 2025-05-30 15:00:00


  0% (0 of 89707) |                      | Elapsed Time: 0:00:00 ETA:  --:--:--


AttributeError: Attribute 'portfolio' not found.

In [ ]:

# 定义交易策略
def kd_strategy(ctx: ExecContext):
    kd_k = ctx.data('kd_k')
    kd_d = ctx.data('kd_d')
    symbol = ctx.symbol
    position = ctx.portfolio.position(symbol)

    # 开仓条件
    if kd_k < 20 and kd_d < 20 and kd_k > kd_d and position.shares == 0:
        # 计算最大仓位
        eligible_symbols = [sym for sym in ctx.symbols if 
                            ctx.data('kd_k', symbol=sym) < 20 and 
                            ctx.data('kd_d', symbol=sym) < 20 and 
                            ctx.data('kd_k', symbol=sym) > ctx.data('kd_d', symbol=sym) and 
                            ctx.portfolio.position(sym).shares == 0]
        max_position_per_symbol = len(eligible_symbols) / 0.8
        # 开仓
        ctx.buy(max_position_per_symbol)
    elif kd_k > 80 and kd_d > 80 and kd_k < kd_d and position.shares == 0:
        # 计算最大仓位
        eligible_symbols = [sym for sym in ctx.symbols if 
                            ctx.data('kd_k', symbol=sym) > 80 and 
                            ctx.data('kd_d', symbol=sym) > 80 and 
                            ctx.data('kd_k', symbol=sym) < ctx.data('kd_d', symbol=sym) and 
                            ctx.portfolio.position(sym).shares == 0]
        max_position_per_symbol = len(eligible_symbols) / 0.8
        # 开仓
        ctx.sell(max_position_per_symbol)

    # 止损条件
    if position.shares > 0 and kd_k < 20 and kd_d < 20 and kd_k < kd_d:
        ctx.sell(position.shares)
    elif position.shares < 0 and kd_k > 80 and kd_d > 80 and kd_k > kd_d:
        ctx.buy(-position.shares)

    # 止盈条件
    if position.shares > 0 and kd_k > 80 and kd_d > 80:
        ctx.sell(position.shares)
    elif position.shares < 0 and kd_k > 20 and kd_d > 20:
        ctx.buy(-position.shares)

# 配置策略
config = StrategyConfig(initial_cash=1000000)
strategy = Strategy(df_sorted, start_date=df_sorted['date'].min(), end_date=df_sorted['date'].max(), config=config)

# 添加执行函数和交易品种
strategy.add_execution(kd_strategy, symbols=df_sorted['symbol'].unique().tolist())

# 运行策略
result = strategy.backtest()

# 打印回测结果
print(result.metrics())

In [ ]:

# 定义交易策略
def kd_strategy(ctx: ExecContext):
    kd_k = ctx.data('kd_k')
    kd_d = ctx.data('kd_d')
    symbol = ctx.symbol
    position = ctx.portfolio.position(symbol)

    # 开仓条件
    if kd_k < 20 and kd_d < 20 and kd_k > kd_d and position.shares == 0:
        # 计算最大仓位
        eligible_symbols = [sym for sym in ctx.symbols if 
                            ctx.data('kd_k', symbol=sym) < 20 and 
                            ctx.data('kd_d', symbol=sym) < 20 and 
                            ctx.data('kd_k', symbol=sym) > ctx.data('kd_d', symbol=sym) and 
                            ctx.portfolio.position(sym).shares == 0]
        max_position_per_symbol = len(eligible_symbols) / 0.8
        # 开仓
        ctx.buy(max_position_per_symbol)
    elif kd_k > 80 and kd_d > 80 and kd_k < kd_d and position.shares == 0:
        # 计算最大仓位
        eligible_symbols = [sym for sym in ctx.symbols if 
                            ctx.data('kd_k', symbol=sym) > 80 and 
                            ctx.data('kd_d', symbol=sym) > 80 and 
                            ctx.data('kd_k', symbol=sym) < ctx.data('kd_d', symbol=sym) and 
                            ctx.portfolio.position(sym).shares == 0]
        max_position_per_symbol = len(eligible_symbols) / 0.8
        # 开仓
        ctx.sell(max_position_per_symbol)

    # 止损条件
    if position.shares > 0 and kd_k < 20 and kd_d < 20 and kd_k < kd_d:
        ctx.sell(position.shares)
    elif position.shares < 0 and kd_k > 80 and kd_d > 80 and kd_k > kd_d:
        ctx.buy(-position.shares)

    # 止盈条件
    if position.shares > 0 and kd_k > 80 and kd_d > 80:
        ctx.sell(position.shares)
    elif position.shares < 0 and kd_k > 20 and kd_d > 20:
        ctx.buy(-position.shares)

# 配置策略
config = StrategyConfig(initial_cash=1000000)
strategy = Strategy(df_sorted, start_date=df_sorted['date'].min(), end_date=df_sorted['date'].max(), config=config)

# 添加执行函数和交易品种
strategy.add_execution(kd_strategy, symbols=df_sorted['symbol'].unique().tolist())

# 运行策略
result = strategy.backtest()

# 打印回测结果
print(result.metrics())

In [19]:
df_sorted

,time,thscode,open,close,high,low,macd,macdsignal,macdhist,bb_high,bb_mid,bb_low,kd_k,kd_d
0,2022-01-04 09:05:00,AGZL.SHF,4836.0,4838.0,4861.0,4830.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-01-04 09:10:00,AGZL.SHF,4838.0,4840.0,4840.0,4831.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-01-04 09:15:00,AGZL.SHF,4841.0,4822.0,4845.0,4818.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-04 09:20:00,AGZL.SHF,4822.0,4819.0,4822.0,4814.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-01-04 09:25:00,AGZL.SHF,4818.0,4814.0,4818.0,4806.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2355044,2025-05-30 14:40:00,ZNZL.SHF,22255.0,22245.0,22260.0,22235.0,-6.641569,-11.538710,4.897140,22287.015551,22232.75,22178.484449,65.952381,49.523810
2355045,2025-05-30 14:45:00,ZNZL.SHF,22245.0,22240.0,22250.0,22235.0,-5.131194,-10.257206,5.126013,22283.854560,22231.50,22179.145440,66.917293,61.181426
2355046,2025-05-30 14:50:00,ZNZL.SHF,22240.0,22195.0,22245.0,22190.0,-7.479119,-9.701589,2.222470,22279.274266,22227.75,22176.225734,43.372320,58.747331
2355047,2025-05-30 14:55:00,ZNZL.SHF,22195.0,22235.0,22245.0,22195.0,-6.042547,-8.969781,2.927234,22268.127717,22225.00,22181.872283,37.816764,49.368792


In [ ]:
def buy_kd_cross(ctx: ExecContext):
    if not ctx.long_pos() and not ctx.short_pos() and ctx.data.kd_k < 20 and ctx.data.kd_d < 20 and ctx.data.kd_k > ctx.data.kd_d:
        if d

In [30]:
import pybroker
from pybroker import ExecContext, StrategyConfig, Strategy
from pybroker.data import DataSource
import pandas as pd
import numpy as np

class KDStrategy(Strategy):
    def __init__(self, config: StrategyConfig):
        super().__init__(config)
        # 存储每个品种的持仓状态
        self.positions = {}  # key: thscode, value: 'long', 'short' or None
        # 最大总仓位限制
        self.max_total_position = 0.8
        # 存储品种列表
        self.thscodes = []
    
    def init(self, context: ExecContext):
        """策略初始化"""
        # 加载预处理好的数据集
        self.df = pd.read_csv(r"C:\Users\Administrator\OneDrive\同步\iFinD data\future_data_merged_all.csv")
        self.df['time'] = pd.to_datetime(self.df['time'])
        self.df = self.df.sort_values(by=['thscode', 'time'])
        
        # 确保指标已计算
        if 'kd_k' not in self.df.columns or 'kd_d' not in self.df.columns:
            raise ValueError("请先计算KD指标")
        
        # 初始化每个品种的持仓状态和品种列表
        self.thscodes = self.df['thscode'].unique()
        for thscode in self.thscodes:
            self.positions[thscode] = None
    
    def handle_bar(self, context: ExecContext):
        """处理每个bar的市场数据"""
        current_time = context.current_time
        # 假设收盘前执行策略（根据实际交易时间调整）
        if current_time.hour == 15 and current_time.minute == 0:
            # 第一步：计算当前符合开仓条件的品种数量
            eligible_count = 0
            for thscode in self.thscodes:
                # 筛选当前品种的最新数据
                df_symbol = self.df[self.df['thscode'] == thscode].tail(1)
                if df_symbol.empty:
                    continue
                    
                kd_k = df_symbol['kd_k'].values[0]
                kd_d = df_symbol['kd_d'].values[0]
                
                # 检查开仓条件
                has_no_position = self.positions[thscode] is None
                long_condition = (kd_k < 20 and kd_d < 20 and kd_k > kd_d and has_no_position)
                short_condition = (kd_k > 80 and kd_d > 80 and kd_k < kd_d and has_no_position)
                
                if long_condition or short_condition:
                    eligible_count += 1
            
            # 第二步：处理每个品种的交易信号
            for thscode in self.thscodes:
                df_symbol = self.df[self.df['thscode'] == thscode].tail(1)
                if df_symbol.empty:
                    continue
                    
                kd_k = df_symbol['kd_k'].values[0]
                kd_d = df_symbol['kd_d'].values[0]
                close_price = df_symbol['close'].values[0]
                
                # 获取当前持仓状态
                position = self.positions[thscode]
                
                # 计算当前品种应占仓位
                if eligible_count > 0:
                    target_position_per_symbol = self.max_total_position / eligible_count
                else:
                    target_position_per_symbol = 0
                
                # 1. 处理多头持仓的止损止盈
                if position == 'long':
                    # 止损条件：kd_k和kd_d均小于20且kd_k小于kd_d
                    stop_loss_condition = (kd_k < 20 and kd_d < 20 and kd_k < kd_d)
                    # 止盈条件：kd_k和kd_d均大于80
                    take_profit_condition = (kd_k > 80 and kd_d > 80)
                    
                    if stop_loss_condition or take_profit_condition:
                        # 平掉所有多头仓位
                        context.order(thscode, 0, price=close_price, side=pybroker.Side.SELL)
                        self.positions[thscode] = None
                        self.log(f"平掉多头仓位: {thscode}, kd_k={kd_k}, kd_d={kd_d}")
                
                # 2. 处理空头持仓的止损止盈
                elif position == 'short':
                    # 止损条件：kd_k和kd_d均大于80且kd_k大于kd_d
                    stop_loss_condition = (kd_k > 80 and kd_d > 80 and kd_k > kd_d)
                    # 止盈条件：kd_k和kd_d均小于20
                    take_profit_condition = (kd_k < 20 and kd_d < 20)
                    
                    if stop_loss_condition or take_profit_condition:
                        # 平掉所有空头仓位
                        context.order(thscode, 0, price=close_price, side=pybroker.Side.BUY)
                        self.positions[thscode] = None
                        self.log(f"平掉空头仓位: {thscode}, kd_k={kd_k}, kd_d={kd_d}")
                
                # 3. 处理开仓条件
                if position is None:
                    # 多头开仓条件
                    long_condition = (kd_k < 20 and kd_d < 20 and kd_k > kd_d)
                    if long_condition:
                        # 计算目标仓位
                        target_shares = int(context.account.cash / close_price * target_position_per_symbol)
                        if target_shares > 0:
                            context.order(thscode, target_shares, price=close_price, side=pybroker.Side.BUY)
                            self.positions[thscode] = 'long'
                            self.log(f"开多仓: {thscode}, 仓位: {target_position_per_symbol:.2f}, kd_k={kd_k}, kd_d={kd_d}")
                    
                    # 空头开仓条件
                    short_condition = (kd_k > 80 and kd_d > 80 and kd_k < kd_d)
                    if short_condition:
                        # 计算目标仓位
                        target_shares = int(context.account.cash / close_price * target_position_per_symbol)
                        if target_shares > 0:
                            context.order(thscode, target_shares, price=close_price, side=pybroker.Side.SELL)
                            self.positions[thscode] = 'short'
                            self.log(f"开空仓: {thscode}, 仓位: {target_position_per_symbol:.2f}, kd_k={kd_k}, kd_d={kd_d}")
    
    def on_trade(self, context: ExecContext, trade: pybroker.Trade):
        """交易执行后的回调"""
        self.log(f"交易完成: {trade.symbol}, 数量: {trade.shares}, 价格: {trade.price}, 方向: {trade.side}")

# 先加载数据获取品种列表
df = pd.read_csv(r"C:\Users\Administrator\OneDrive\同步\iFinD data\future_data_merged_all.csv")
df['time'] = pd.to_datetime(df['time'])
thscodes = df['thscode'].unique()

# 策略配置 - 使用start_time和end_time
config = StrategyConfig(
    initial_cash=3000000,  # 初始资金
    start_time="2023-01-01 09:30:00",  # 修改为start_time
    end_time="2023-12-31 15:00:00",    # 修改为end_time
)

# 运行策略
strategy = KDStrategy(config)
results = strategy.run()

# 打印策略结果
print(f"策略最终资产: {results.final_balance}")
print(f"总收益率: {results.total_return:.2f}%")
print(f"最大回撤: {results.max_drawdown:.2f}%")

TypeError: StrategyConfig.__init__() got an unexpected keyword argument 'start_time'